# Dunnhumby — M3 중심화 가치그래프 (가치축 / 가치축+반복거래축)

**무엇을 바꾸는가.** user–item 엣지의 가중치만 바꿉니다. 임베딩·손실함수·평가는 M1과 같습니다.
가중치는 `exp(beta * [q_V(u)*z_V(u,i) + gamma * q_N(u)*z_N(u,i)])`이고, `z`는 그 고객의
지출(가치축) 또는 바구니 수(반복거래축)에서 해당 상품이 차지하는 **로그 비중**을 고객 내에서
중심화하고 다시 상품 내에서 중심화한 값입니다.

**왜 두 번 중심화하는가.** 예전 M3(`vg_value`: `w = 1 + α·log(1+지출/평균가)`)는 고객과 무관하게
비싼 상품의 엣지를 전부 키워서, 가중치가 사실상 상품 가격의 대리변수가 됩니다. 두 번 중심화하면
"이 고객이 자기 평균보다, 그리고 그 상품의 일반적 수준보다 더 많이 쓴 곳"만 남습니다.

**arm 두 개.** A는 가치축만(`gamma=0`), B는 반복거래축을 함께 씁니다(`gamma=1`). 두 축을 서로
같은 크기로 맞추지 않으므로, 반복구매가 거의 없는 데이터에서는 반복거래축이 스스로 작아집니다
(Dunnhumby 활동/가치 표준편차 비율 0.488, H&M 0.340). B−A가 반복거래축 자체의 기여입니다.

**강도 knob은 beta 하나**이고, 결과가 아니라 "엣지 가중치 변동계수 0.20"이라는 데이터 통계로
역산합니다. 데이터 성격에 따라 값이 저절로 달라집니다(Dunnhumby A 0.390, H&M A 0.800).

**학습 전 게이트.** 가중치가 가격·인기·degree의 대리변수가 되면 학습하지 않고 멈춥니다. 로컬에서
두 데이터셋 모두 통과를 확인했습니다(상품가중↔가격 Dunnhumby −0.000/−0.005, H&M −0.006/+0.002).

**M1은 seed 42만 용량탐색 곡선을 재사용합니다** — 용량탐색이 seed 42만 돌렸으므로 43·44의 M1은 명시적 허용 아래 같은 경로로 새로 학습합니다. seed 42/43/44, 300 epoch, 25마다 평가.
판독은 100·300 epoch 두 지점에서 arm−M1과 B−A이며, 결과를 본 뒤 epoch이나 arm을 고르지 않습니다.

반복 노출된 개발분할(학습 ≤683, 개발 684–690) 3 seed이므로 유의성·일반화·CLV 귀속은 주장하지
않습니다. test/holdout은 계산하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
M1_DIR = ROOT/'results_v3_dunnhumby_clv_m2_capacity_search_v1'
if not M1_DIR.is_dir():
    if os.path.ismount('/content/drive'):
        raise RuntimeError(f'Drive는 연결됐지만 M1 용량탐색 결과가 없습니다: {M1_DIR}. 계정/경로 확인. 새 학습 안 함.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습은 시작되지 않았습니다.') from exc
    if not M1_DIR.is_dir():
        raise RuntimeError(f'M1 용량탐색 결과가 없습니다: {M1_DIR}. M1을 재학습하지 않고 중단합니다.')

SOURCE_COMMIT = 'b6c579fa869c3e383368ea473cb4351f9ff1beb4'
REPO = Path('/content/clv-m3-centered-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), \
        '다른 소스가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import lightgcn_clv_m3_centered_value_graph as m3

# 1단계는 SEEDS=(42,)로 돌린다. 끝나고 판단하지 말고, 그대로 (42,43,44)로 바꿔 다시 실행하면
# 끝난 학습은 건너뛰고 이어진다. 판정은 3 seed 전부 모인 뒤에만 한다.
SEEDS = (42,)
ALLOW_M1 = tuple(f'{s}:m1' for s in SEEDS if s != 42)   # 용량탐색은 seed 42만 돌렸다
cfg = m3.configure_centered_graph(
    m1_result_dir=str(M1_DIR), seeds=SEEDS, allow_baseline_training=ALLOW_M1)
print('이번 실행 seed:', cfg.seeds, '| 새로 학습할 M1:', cfg.allow_baseline_training or '없음')
print(json.dumps(m3.preflight_summary(cfg), ensure_ascii=False, indent=2))

## 1. M1 곡선 확보 (seed 42는 재사용, 43·44는 새로 학습)

용량탐색은 **seed 42만** 돌렸으므로 43·44의 M1 300-epoch 곡선이 없다. 이 두 개는 용량탐색 자신의
M1 경로로(같은 config hash·파일명·프로토콜) 새로 학습하며, `allow_baseline_training`에 명시한
seed만 학습한다. 명시하지 않은 seed는 여기서 멈춘다 — M1을 몰래 다시 학습하지 않는다.

**비용:** seed 42는 M3 두 개만(약 3.5시간). seed 43·44를 더하면 M1 2개 + M3 4개(약 10시간)가
추가된다. 총 8회 × 300 epoch ≈ 13~14시간이며, 끊겨도 같은 설정으로 다시 실행하면 이어진다.

In [ ]:
# 이번 seed의 M1 곡선을 확인한다. 허용 목록에 있는 seed만 새로 학습하고, 그 외에는 멈춘다.
trained = m3.train_missing_baselines(cfg)
print('새로 학습한 M1 seed:', trained or '없음 (전부 재사용)')
baselines = m3.load_baseline_curves(cfg)
print('사용하는 M1 곡선:', {seed: len(curve) for seed, curve in baselines.items()})
assert set(baselines) == set(cfg.seeds), f'M1 곡선이 없는 seed: {set(cfg.seeds) - set(baselines)}'

## 2. 엣지 가중치 게이트 (학습 없음)
beta를 역산하고 가중치를 감사합니다. 가격·인기·degree 상관이 한계를 넘으면 `build_arm_graph`가
예외를 던지고 학습은 시작되지 않습니다. 로컬 확인값은 아래와 같습니다 — Colab 수치가 크게
다르면 입력이 달라진 것이므로 멈추고 확인하세요.

| arm | beta | CV | 상품↔가격 | 상품↔인기 | 고객↔degree |
|---|---|---|---|---|---|
| A 가치축만 | 0.390 | 0.200 | −0.000 | −0.025 | +0.010 |
| B 가치+반복거래 | 0.216 | 0.200 | −0.005 | −0.014 | +0.012 |

In [ ]:
import pandas as pd
prepared = m3._prepare(cfg)
graphs = {}
for spec in m3.arm_specifications():
    graphs[spec['model_id']] = m3.build_arm_graph(prepared, cfg, spec)   # 게이트 실패 시 예외
print(pd.DataFrame([{'arm': spec['arm'], 'beta': graphs[spec['model_id']]['beta'],
                     **graphs[spec['model_id']]['audit']}
                    for spec in m3.arm_specifications()]).to_string(index=False))

## 3. 학습 (이번 seed의 arm 2개, 필요하면 M1도)

epoch마다 Drive에 저장하므로 끊긴 뒤 같은 설정으로 다시 실행하면 이어진다. 완료된 arm·seed는
건너뛴다. `SEEDS`를 (42,)→(42,43,44)로 바꿔 다시 실행하는 것이 정상 경로다.

이전 커밋에서 중간에 죽은 실행이 남긴 재개파일은 자동으로 지우고 그 arm만 처음부터 학습한다(무엇을 지웠는지 출력된다). 재개 신원에 소스 커밋이 들어가므로 코드가 바뀐 뒤의 체크포인트는 이어쓸 수 없다 — 끝난 결과 파일은 지우지 않는다.

In [ ]:
import torch
assert torch.cuda.is_available(), '전체 실험은 GPU 런타임을 사용하세요.'
curve = m3.run_centered_graph(cfg)   # 필요한 M1 학습까지 포함한다

## 4. 판독표와 원본 ZIP

`difference`는 100·300 epoch에서 arm−M1과 B−A다. 참조 곡선이 없으면 조용히 빠지지 않고 예외가
난다. **seed 42만 돌린 1단계 결과로는 판정하지 않는다** — 같은 분할에서 M2−M1 Recall@10 차이의
seed 간 표준편차가 0.000384였으므로 한 seed는 방향을 정할 근거가 못 된다. 1단계는 "설계가
크게 망가지지 않았는지"만 본다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
difference = m3.difference_frame(curve)
paths = curve.attrs['result_paths']
metrics = ['recall@10', 'ndcg@10', 'recall@50', 'ndcg@50',
           'price_purchase_amount_weighted_hit@10', 'vndcg@10',
           'price_purchase_amount_weighted_hit@50', 'vndcg@50']
print(difference.groupby(['model_id', 'reference', 'epoch'])[metrics].mean().to_string())
print()
print(json.dumps(curve.attrs['reading'], ensure_ascii=False, indent=2))

zip_path = Path('/content/m3_centered_value_graph_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))